# Project 05 — Experiment 2: Curated Dataset V3, Retraining, and Experiment Comparison

This notebook preserves the completed first experiment, builds a higher-quality
instruction dataset without using the held-out benchmark as training data,
trains a new FLAN-T5-base LoRA adapter, evaluates it on the same 80-prompt
benchmark, and compares Experiment 2 directly with Experiment 1.

**Release rule:** Experiment 1 is archived, not deleted. Experiment 2 is not
promoted until automated comparison and human factual review are complete.

## 0. Resolve the project root

In [1]:
from pathlib import Path
from datetime import datetime
import json
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT = PROJECT_ROOT.resolve()
sys.path.insert(0, str(PROJECT_ROOT))

assert (PROJECT_ROOT / "src").exists(), "Open Jupyter from Project 05 or its notebooks folder."
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm


## 1. Experiment controls and stable output paths

In [2]:
EXPERIMENT1_RUN_NAME = "flan_t5_base_lora_20260730_120312"
EXPERIMENT1_DIR = PROJECT_ROOT / "outputs" / "experiments" / EXPERIMENT1_RUN_NAME
assert EXPERIMENT1_DIR.exists(), f"Experiment 1 was not found: {EXPERIMENT1_DIR}"

ACTIVE_RUN_FILE = PROJECT_ROOT / "outputs" / "experiment2_active_run.json"
if ACTIVE_RUN_FILE.exists():
    active_run = json.loads(ACTIVE_RUN_FILE.read_text(encoding="utf-8"))
    EXPERIMENT2_RUN_NAME = active_run["run_name"]
else:
    EXPERIMENT2_RUN_NAME = f"flan_t5_base_lora_exp2_v3_{datetime.now():%Y%m%d_%H%M%S}"
    ACTIVE_RUN_FILE.parent.mkdir(parents=True, exist_ok=True)
    ACTIVE_RUN_FILE.write_text(
        json.dumps({"run_name": EXPERIMENT2_RUN_NAME}, indent=2),
        encoding="utf-8",
    )

EXPERIMENT2_DIR = PROJECT_ROOT / "outputs" / "experiments" / EXPERIMENT2_RUN_NAME
EXPERIMENT2_DIR.mkdir(parents=True, exist_ok=True)

CREATE_FULL_EXPERIMENT1_ZIP = True
REUSE_EXPERIMENT1_TEACHER_RECORDS = False
RUN_TRAINING = True
RUN_EVALUATION = True
PROMOTE_EXPERIMENT2 = False

print("Experiment 1:", EXPERIMENT1_DIR)
print("Experiment 2:", EXPERIMENT2_DIR)
print("Reuse Experiment 1 teacher records:", REUSE_EXPERIMENT1_TEACHER_RECORDS)

Experiment 1: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_20260730_120312
Experiment 2: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452
Reuse Experiment 1 teacher records: False


## 2. Preserve Experiment 1

This creates checksums, an experiment card, a summary, a notebook snapshot, and
an optional full ZIP. The original folder remains exactly where it is.

In [3]:
from src.experiment_archive import archive_experiment

archive_result = archive_experiment(
    EXPERIMENT1_DIR,
    PROJECT_ROOT / "outputs" / "experiment_archives",
    archive_label="experiment_1_initial_lora",
    create_full_zip=CREATE_FULL_EXPERIMENT1_ZIP,
    extra_files=[
        PROJECT_ROOT / "notebooks" / "05_full_training_evaluation_pipeline.ipynb"
    ],
)
display(archive_result)

assert EXPERIMENT1_DIR.exists(), "The original Experiment 1 folder must remain."
assert archive_result["source_still_exists"] is True

{'status': 'archived_without_deleting_source',
 'source_experiment': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\outputs\\experiments\\flan_t5_base_lora_20260730_120312',
 'source_still_exists': True,
 'archive_directory': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\outputs\\experiment_archives\\experiment_1_initial_lora',
 'full_zip': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\outputs\\experiment_archives\\experiment_1_initial_lora\\experiment_1_initial_lora_full.zip',
 'summary_file': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\outputs\\experiment_archives\\experiment_1_initial_lora\\experiment_1_summary.json',
 'card_file': 'C:\\Users\\atri

## 3. Summarize the weaknesses found in Experiment 1

In [4]:
import pandas as pd

exp1_lora_review_path = (
    EXPERIMENT1_DIR / "evaluation" / "lora_model" / "manual_review_results.csv"
)
exp1_comparison_review_path = (
    EXPERIMENT1_DIR / "evaluation" / "comparison" / "per_example_base_vs_lora.csv"
)
assert exp1_lora_review_path.exists()
assert exp1_comparison_review_path.exists()

exp1_review = pd.read_csv(exp1_lora_review_path)
rating_columns = [
    "human_factuality_1_to_5",
    "human_relevance_1_to_5",
    "human_clarity_1_to_5",
    "human_instruction_following_1_to_5",
]

weakness_by_topic = (
    exp1_review
    .groupby(["topic", "category"], dropna=False)[rating_columns]
    .mean()
    .reset_index()
    .sort_values(
        ["human_factuality_1_to_5", "human_instruction_following_1_to_5"]
    )
)
display(weakness_by_topic.head(30))

weakness_output = EXPERIMENT2_DIR / "experiment1_weakness_summary.csv"
weakness_by_topic.to_csv(weakness_output, index=False)
print("Saved:", weakness_output)

,topic,category,human_factuality_1_to_5,human_relevance_1_to_5,human_clarity_1_to_5,human_instruction_following_1_to_5
1,BERTScore,Quality analytics,1.0,2.0,2.0,1.0
3,F1-score,Quality analytics,1.0,2.0,1.0,1.0
5,LoRA,Quality analytics,1.0,2.0,2.0,1.0
7,MAE,Quality analytics,1.0,2.0,2.0,1.0
9,PR-AUC,Quality analytics,1.0,2.0,2.0,1.0
10,R-squared,Metric explanation,1.0,2.0,2.0,1.0
13,RMSE,Quality analytics,1.0,2.0,2.0,1.0
15,ROC-AUC,Quality analytics,1.0,2.0,2.0,1.0
17,attention,Quality analytics,1.0,2.0,2.0,1.0
20,class imbalance,Concept explanation,1.0,2.0,2.0,1.0


Saved: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\experiment1_weakness_summary.csv


## 4. Build the Version 3 curated dataset

The default intentionally does **not** reuse the weak teacher-generated records
from Experiment 1. It combines the original self-authored seed set with
expert-curated topic cards, comparisons, code examples, and workflow answers.
The benchmark is used only for leakage screening.

In [5]:
from src.experiment2_dataset import build_dataset_v3

DATASET_V3_PATH = PROJECT_ROOT / "data" / "ml_ds_instruction_dataset_v3.jsonl"
DATASET_V3_REPORT_DIR = EXPERIMENT2_DIR / "dataset_build"

dataset_v3_result = build_dataset_v3(
    seed_dataset_path=PROJECT_ROOT / "data" / "ml_ds_instruction_dataset.jsonl",
    previous_v2_path=PROJECT_ROOT / "data" / "ml_ds_instruction_dataset_v2.jsonl",
    benchmark_path=PROJECT_ROOT / "data" / "benchmark_prompts_v2.jsonl",
    topic_cards_path=PROJECT_ROOT / "data" / "curated_topic_cards_v3.json",
    comparisons_path=PROJECT_ROOT / "data" / "curated_comparisons_v3.json",
    code_examples_path=PROJECT_ROOT / "data" / "curated_code_examples_v3.json",
    workflows_path=PROJECT_ROOT / "data" / "curated_workflows_v3.json",
    rules_path=PROJECT_ROOT / "data" / "experiment2_quality_rules.json",
    output_dataset_path=DATASET_V3_PATH,
    output_report_dir=DATASET_V3_REPORT_DIR,
    reuse_teacher_records=REUSE_EXPERIMENT1_TEACHER_RECORDS,
    seed=52,
)
display(dataset_v3_result)

assert dataset_v3_result["report"]["final_records"] >= 400
assert dataset_v3_result["report"]["validation_records"] > 0
assert dataset_v3_result["report"]["test_records"] > 0

{'status': 'completed',
 'dataset_path': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\data\\ml_ds_instruction_dataset_v3.jsonl',
 'reuse_teacher_records': False,
 'report': {'seed_records_loaded': 93,
  'previous_v2_records_loaded': 543,
  'previous_v2_records_reused': 0,
  'curated_topic_records': 275,
  'curated_comparison_records': 40,
  'curated_code_records': 20,
  'curated_workflow_records': 15,
  'rejected_quality': 0,
  'rejected_benchmark_overlap': 0,
  'rejected_near_duplicate': 0,
  'final_records': 442,
  'train_records': 352,
  'validation_records': 45,
  'test_records': 45,
  'output_word_mean': 49.03,
  'output_word_median': 48.0,
  'source_counts': {'expert-curated-v3-topic-card': 275,
   'expert-curated-v3-workflow': 14,
   'expert-curated-v3-comparison': 40,
   'expert-curated-v3-code': 20,
   'self-authored-synthetic': 93},
  'category_counts': {'Example generation': 61,
   'Data Sc

## 5. Review the Version 3 dataset and approve it

In [6]:
dataset_v3_df = pd.read_json(DATASET_V3_PATH, lines=True)
print("Dataset shape:", dataset_v3_df.shape)

display(
    dataset_v3_df.groupby(["category", "difficulty"])
    .size()
    .rename("examples")
    .reset_index()
    .sort_values(["category", "difficulty"])
)

review_sample_path = DATASET_V3_REPORT_DIR / "dataset_v3_review_sample.csv"
review_sample = pd.read_csv(review_sample_path)
display(
    review_sample[
        ["instruction", "output", "category", "difficulty", "topic", "source", "split"]
    ]
)

print("Review sample file:", review_sample_path)
print("Open it in Excel if a wider view is easier.")

Dataset shape: (442, 9)


,category,difficulty,examples
0,Algorithm comparison,intermediate,30
1,Beginner-friendly explanation,beginner,61
2,Concept explanation,advanced,2
3,Concept explanation,beginner,16
4,Concept explanation,intermediate,72
5,Data Science workflow,advanced,3
6,Data Science workflow,intermediate,9
7,Example generation,beginner,6
8,Example generation,intermediate,55
9,Interview-style answer,advanced,20


,instruction,output,category,difficulty,topic,source,split
0,Compare random forest and gradient boosting.,Random forest builds many independent trees an...,Algorithm comparison,intermediate,ensemble learning,self-authored-synthetic,train
1,Contrast ROC-AUC with PR-AUC and give a decisi...,Core difference: ROC-AUC summarizes ranking ac...,Algorithm comparison,intermediate,ROC-AUC vs PR-AUC,expert-curated-v3-comparison,validation
2,Compare clustering and classification.,Clustering discovers groups without target lab...,Algorithm comparison,intermediate,learning paradigms,self-authored-synthetic,test
3,Contrast overfitting with underfitting and giv...,Core difference: Overfitting captures training...,Algorithm comparison,intermediate,overfitting vs underfitting,expert-curated-v3-comparison,train
4,Explain data drift to a beginner without using...,The model sees different feature values or pop...,Beginner-friendly explanation,beginner,data drift,expert-curated-v3-topic-card,train
5,Explain Transformer to a beginner without usin...,Each token can directly combine information fr...,Beginner-friendly explanation,beginner,Transformer,expert-curated-v3-topic-card,test
6,Explain full fine-tuning to a beginner without...,"The entire model can adapt, providing high cap...",Beginner-friendly explanation,beginner,full fine-tuning,expert-curated-v3-topic-card,train
7,Explain model deployment to a beginner without...,"Deployment connects model artifacts, preproces...",Beginner-friendly explanation,beginner,model deployment,expert-curated-v3-topic-card,train
8,Teach the core idea of L1 regularization using...,Definition: L1 regularization adds the absolut...,Concept explanation,intermediate,L1 regularization,expert-curated-v3-topic-card,validation
9,Teach the core idea of feature selection using...,Definition: Feature selection chooses a subset...,Concept explanation,intermediate,feature selection,expert-curated-v3-topic-card,validation


Review sample file: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\dataset_build\dataset_v3_review_sample.csv
Open it in Excel if a wider view is easier.


In [7]:
import os

dataset_review_dir = EXPERIMENT2_DIR / "dataset_build"

print("Dataset review directory:")
print(dataset_review_dir)

print("\nReview sample:")
print(dataset_review_dir / "dataset_v3_review_sample.csv")

print("\nQuality report:")
print(dataset_review_dir / "dataset_v3_quality_report.json")

print("\nRejection log:")
print(dataset_review_dir / "dataset_v3_rejection_log.json")

os.startfile(str(dataset_review_dir))

Dataset review directory:
C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\dataset_build

Review sample:
C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\dataset_build\dataset_v3_review_sample.csv

Quality report:
C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\dataset_build\dataset_v3_quality_report.json

Rejection log:
C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\dataset_build\dataset_v3_rejection_log.json


In [8]:
import json
import pandas as pd

review_path = dataset_review_dir / "dataset_v3_review_sample.csv"
quality_report_path = dataset_review_dir / "dataset_v3_quality_report.json"
rejection_log_path = dataset_review_dir / "dataset_v3_rejection_log.json"

review_df = pd.read_csv(review_path)

print("=" * 70)
print("DATASET V3 REVIEW SAMPLE")
print("=" * 70)
print(f"Review records: {len(review_df)}")
print(f"Columns: {review_df.columns.tolist()}")

display(review_df)

DATASET V3 REVIEW SAMPLE
Review records: 60
Columns: ['id', 'instruction', 'input', 'output', 'category', 'difficulty', 'topic', 'source', 'split']


,id,instruction,input,output,category,difficulty,topic,source,split
0,ml_ds_v3_0109,Compare random forest and gradient boosting.,NaN,Random forest builds many independent trees an...,Algorithm comparison,intermediate,ensemble learning,self-authored-synthetic,train
1,ml_ds_v3_0228,Contrast ROC-AUC with PR-AUC and give a decisi...,"Use the headings Core difference, Prefer the f...",Core difference: ROC-AUC summarizes ranking ac...,Algorithm comparison,intermediate,ROC-AUC vs PR-AUC,expert-curated-v3-comparison,validation
2,ml_ds_v3_0257,Compare clustering and classification.,NaN,Clustering discovers groups without target lab...,Algorithm comparison,intermediate,learning paradigms,self-authored-synthetic,test
3,ml_ds_v3_0223,Contrast overfitting with underfitting and giv...,"Use the headings Core difference, Prefer the f...",Core difference: Overfitting captures training...,Algorithm comparison,intermediate,overfitting vs underfitting,expert-curated-v3-comparison,train
4,ml_ds_v3_0312,Explain data drift to a beginner without using...,Include a practical example and one limitation...,The model sees different feature values or pop...,Beginner-friendly explanation,beginner,data drift,expert-curated-v3-topic-card,train
5,ml_ds_v3_0139,Explain Transformer to a beginner without usin...,Include a practical example and one limitation...,Each token can directly combine information fr...,Beginner-friendly explanation,beginner,Transformer,expert-curated-v3-topic-card,test
6,ml_ds_v3_0235,Explain full fine-tuning to a beginner without...,Include a practical example and one limitation...,"The entire model can adapt, providing high cap...",Beginner-friendly explanation,beginner,full fine-tuning,expert-curated-v3-topic-card,train
7,ml_ds_v3_0441,Explain model deployment to a beginner without...,Include a practical example and one limitation...,"Deployment connects model artifacts, preproces...",Beginner-friendly explanation,beginner,model deployment,expert-curated-v3-topic-card,train
8,ml_ds_v3_0368,Teach the core idea of L1 regularization using...,Write for an intermediate ML learner and keep ...,Definition: L1 regularization adds the absolut...,Concept explanation,intermediate,L1 regularization,expert-curated-v3-topic-card,validation
9,ml_ds_v3_0131,Teach the core idea of feature selection using...,Write for an intermediate ML learner and keep ...,Definition: Feature selection chooses a subset...,Concept explanation,intermediate,feature selection,expert-curated-v3-topic-card,validation


In [9]:
print("Examples by category:")

display(
    review_df["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="examples")
)

print("\nExamples by difficulty:")

display(
    review_df["difficulty"]
    .value_counts()
    .rename_axis("difficulty")
    .reset_index(name="examples")
)

Examples by category:


,category,examples
0,Interview-style answer,15
1,Small code example,10
2,ML project guidance,6
3,Concept explanation,5
4,Algorithm comparison,4
5,Beginner-friendly explanation,4
6,Data Science workflow,4
7,Example generation,4
8,Metric explanation,4
9,Quality analytics,4



Examples by difficulty:


,difficulty,examples
0,intermediate,29
1,advanced,26
2,beginner,5


In [10]:
important_pattern = (
    "classification|regression|data leakage|cross-validation|"
    "standardization|precision|recall|f1|transformer|attention|"
    "lora|peft|quality analytics|code"
)

important_examples = review_df[
    review_df["topic"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains(important_pattern, regex=True)
]

display(
    important_examples.reset_index(drop=True)
)

,id,instruction,input,output,category,difficulty,topic,source,split
0,ml_ds_v3_0139,Explain Transformer to a beginner without usin...,Include a practical example and one limitation...,Each token can directly combine information fr...,Beginner-friendly explanation,beginner,Transformer,expert-curated-v3-topic-card,test
1,ml_ds_v3_0004,Outline a leakage-safe workflow for building a...,Present actionable steps and include evaluatio...,"1. Define the prediction time, positive class,...",Data Science workflow,advanced,quality classification workflow,expert-curated-v3-workflow,train
2,ml_ds_v3_0256,Answer an interview question about quality ana...,"State what it is, when it is useful, and one c...",Quality analytics classification predicts a di...,Interview-style answer,intermediate,quality analytics classification,expert-curated-v3-topic-card,train
3,ml_ds_v3_0288,Answer an interview question about F1-score in...,"State what it is, when it is useful, and one c...",F1-score is the harmonic mean of precision and...,Interview-style answer,intermediate,F1-score,expert-curated-v3-topic-card,train
4,ml_ds_v3_0271,Describe a safe ML approach for root-cause pri...,NaN,"Use historical, non-confidential features to r...",Quality analytics,intermediate,quality analytics,self-authored-synthetic,train
5,ml_ds_v3_0263,"Give a concrete, non-confidential quality anal...","Use a manufacturing, inspection, complaint, or...",Quality use case: Report F1 together with the ...,Quality analytics,intermediate,F1-score,expert-curated-v3-topic-card,train
6,ml_ds_v3_0068,Show a PEFT LoRA configuration for a T5 sequen...,"Use Python and explain the leakage, evaluation...","```python from peft import LoraConfig, TaskTyp...",Small code example,advanced,LoRA configuration,expert-curated-v3-code,train
7,ml_ds_v3_0047,Give a portfolio-ready interview answer compar...,Do not claim that one is universally better.,Precision measures correctness among predicted...,Interview-style answer,advanced,precision vs recall,expert-curated-v3-comparison,train
8,ml_ds_v3_0020,Give a portfolio-ready interview answer compar...,Do not claim that one is universally better.,"A fixed split uses one validation partition, w...",Interview-style answer,advanced,train validation test split vs cross-validation,expert-curated-v3-comparison,train
9,ml_ds_v3_0241,Create a concise evaluation snippet for precis...,"Use Python and explain the leakage, evaluation...",```python from sklearn.metrics import ( classi...,Small code example,advanced,classification metrics,expert-curated-v3-code,test


In [11]:
with open(quality_report_path, "r", encoding="utf-8") as file:
    quality_report = json.load(file)

display(quality_report)

{'status': 'completed',
 'dataset_path': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\data\\ml_ds_instruction_dataset_v3.jsonl',
 'reuse_teacher_records': False,
 'report': {'seed_records_loaded': 93,
  'previous_v2_records_loaded': 543,
  'previous_v2_records_reused': 0,
  'curated_topic_records': 275,
  'curated_comparison_records': 40,
  'curated_code_records': 20,
  'curated_workflow_records': 15,
  'rejected_quality': 0,
  'rejected_benchmark_overlap': 0,
  'rejected_near_duplicate': 0,
  'final_records': 442,
  'train_records': 352,
  'validation_records': 45,
  'test_records': 45,
  'output_word_mean': 49.03,
  'output_word_median': 48.0,
  'source_counts': {'expert-curated-v3-topic-card': 275,
   'expert-curated-v3-workflow': 14,
   'expert-curated-v3-comparison': 40,
   'expert-curated-v3-code': 20,
   'self-authored-synthetic': 93},
  'category_counts': {'Example generation': 61,
   'Data Sc

In [12]:
with open(rejection_log_path, "r", encoding="utf-8") as file:
    rejection_log = json.load(file)

print("Rejected records:", len(rejection_log))

if rejection_log:
    display(pd.DataFrame(rejection_log).head(50))
else:
    print("No rejected records were recorded.")

Rejected records: 0
No rejected records were recorded.


In [13]:
DATASET_V3_HUMAN_REVIEW_APPROVED = True

if RUN_TRAINING and not DATASET_V3_HUMAN_REVIEW_APPROVED:
    raise RuntimeError(
        "Experiment 2 training is blocked. Review dataset_v3_review_sample.csv, "
        "correct any weak records, then set DATASET_V3_HUMAN_REVIEW_APPROVED = True."
    )

print("Dataset V3 human-review approval passed.")

Dataset V3 human-review approval passed.


## 6. Verify the RTX environment and Experiment 2 training configuration

In [14]:
import torch
from dataclasses import asdict
from src.config import ModelConfig
from src.experiment2_training import Experiment2TrainingConfig
from src.hardware_utils import detect_hardware

hardware = detect_hardware()
assert hardware.cuda_available, "CUDA is required."
assert "RTX 5090" in hardware.gpu_name, f"Unexpected GPU: {hardware.gpu_name}"

model_config = ModelConfig(
    base_model_id="google/flan-t5-base",
    max_input_length=512,
    max_target_length=256,
)
training_config = Experiment2TrainingConfig(
    r=32,
    lora_alpha=64,
    learning_rate=5e-5,
    num_train_epochs=5.0,
    warmup_steps=20,
    early_stopping_patience=2,
    seed=52,
)

display({
    "hardware": hardware.to_dict(),
    "model_config": asdict(model_config),
    "experiment2_training_config": asdict(training_config),
})

{'hardware': {'python_version': '3.12.10',
  'platform': 'Windows-11-10.0.26200-SP0',
  'torch_version': '2.12.1+cu132',
  'cuda_available': True,
  'cuda_version': '13.2',
  'cudnn_version': '92000',
  'gpu_name': 'NVIDIA GeForce RTX 5090',
  'gpu_vram_gb': 31.84,
  'compute_capability': '12.0',
  'bf16_supported': True,
  'recommended_precision': 'bf16',
  'recommended_model_id': 'google/flan-t5-base',
  'train_batch_size': 8,
  'eval_batch_size': 8,
  'gradient_accumulation_steps': 2,
  'gradient_checkpointing': False,
  'dataloader_num_workers': 4},
 'model_config': {'base_model_id': 'google/flan-t5-base',
  'adapter_id': '',
  'local_adapter_path': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\models\\lora_adapter',
  'max_input_length': 512,
  'max_target_length': 256,
  'device': 'auto',
  'torch_dtype': 'auto',
  'trust_remote_code': False},
 'experiment2_training_config': {'r': 32,
  'lora_alp

## 7. Train the Experiment 2 FLAN-T5-base LoRA adapter

In [15]:
import gc
import shutil
import torch

partial_training_dir = EXPERIMENT2_DIR / "training"

print("Incomplete training directory:")
print(partial_training_dir)

if partial_training_dir.exists():
    shutil.rmtree(partial_training_dir)
    print("Incomplete Experiment 2 training directory deleted.")
else:
    print("No incomplete training directory was found.")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("RTX GPU cache cleared.")

Incomplete training directory:
C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\training
Incomplete Experiment 2 training directory deleted.
RTX GPU cache cleared.


In [16]:
from src.experiment2_training import train_experiment2_adapter

TRAINING_DIR = EXPERIMENT2_DIR / "training"

if RUN_TRAINING:
    experiment2_training_metadata = train_experiment2_adapter(
        DATASET_V3_PATH,
        TRAINING_DIR,
        model_config=model_config,
        training_config=training_config,
        hardware_profile=hardware,
    )
else:
    experiment2_training_metadata = json.loads(
        (TRAINING_DIR / "model_metadata.json").read_text(encoding="utf-8")
    )

display(experiment2_training_metadata)

EXPERIMENT2_ADAPTER_PATH = TRAINING_DIR / "lora_adapter"
assert (EXPERIMENT2_ADAPTER_PATH / "adapter_config.json").exists()
assert (EXPERIMENT2_ADAPTER_PATH / "adapter_model.safetensors").exists()
print("Experiment 2 adapter:", EXPERIMENT2_ADAPTER_PATH)

C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2309.19it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 PRETRAINED CHECKPOINT VERIFICATION
Shared embedding loaded       : True
Encoder uses shared embedding : True
Decoder uses shared embedding : True
LM head uses shared embedding : False
Config tie_word_embeddings   : True
Pretrained FLAN-T5 embedding verification passed.


Tokenizing Experiment 2 test split: 100%|██████████| 45/45 [00:00<00:00, 2645.28 examples/s]


trainable params: 3,538,944 || all params: 251,116,800 || trainable%: 1.4093


Epoch,Training Loss,Validation Loss
1,8.742343,4.003652
2,8.561387,3.884036
3,8.400881,3.800303
4,8.380283,3.766414
5,8.195749,3.760525


Training Loss,Validation Loss,Epoch
8.195749,3.760525,5


Training Loss,Validation Loss,Epoch
8.195749,3.808744,5


{'status': 'completed',
 'experiment': 'experiment_2_dataset_quality_upgrade',
 'base_model': 'google/flan-t5-base',
 'fine_tuning_method': 'LoRA/PEFT',
 'adapter_path': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\outputs\\experiments\\flan_t5_base_lora_exp2_v3_20260730_135452\\training\\lora_adapter',
 'tokenizer_path': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\outputs\\experiments\\flan_t5_base_lora_exp2_v3_20260730_135452\\training\\tokenizer',
 'dataset_path': 'C:\\Users\\atripathi\\OneDrive - Veralto\\Desktop\\AI Codes\\GIT Projects\\transformer-projects\\05-instruction-tuned-domain-llm\\data\\ml_ds_instruction_dataset_v3.jsonl',
 'dataset_validation': {'total_records': 442,
  'valid_records': 442,
  'removed_records': 0,
  'empty_instructions': 0,
  'empty_outputs': 0,
  'duplicate_instructions': 0,
  'too_

Experiment 2 adapter: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\training\lora_adapter


## 8. Inspect the Experiment 2 training artifacts

In [17]:
required_training_artifacts = [
    TRAINING_DIR / "model_metadata.json",
    TRAINING_DIR / "training_curve.png",
    TRAINING_DIR / "training_log_history.json",
    TRAINING_DIR / "training_log_history.csv",
    TRAINING_DIR / "lora_adapter" / "adapter_config.json",
    TRAINING_DIR / "lora_adapter" / "adapter_model.safetensors",
]
artifact_status = pd.DataFrame({
    "artifact": [str(path.relative_to(EXPERIMENT2_DIR)) for path in required_training_artifacts],
    "exists": [path.exists() for path in required_training_artifacts],
    "bytes": [path.stat().st_size if path.exists() else 0 for path in required_training_artifacts],
})
display(artifact_status)
assert artifact_status["exists"].all()

,artifact,exists,bytes
0,training\model_metadata.json,True,4728
1,training\training_curve.png,True,47447
2,training\training_log_history.json,True,3614
3,training\training_log_history.csv,True,1832
4,training\lora_adapter\adapter_config.json,True,1112
5,training\lora_adapter\adapter_model.safetensors,True,14176016


## 9. Evaluate Base FLAN-T5 versus Experiment 2

This uses the same held-out `benchmark_prompts_v2.jsonl` and the same evaluation
pipeline used for Experiment 1.

In [18]:
from src.advanced_evaluation import run_base_vs_lora_evaluation
from src.config import EvaluationConfig

EXPERIMENT2_EVALUATION_DIR = EXPERIMENT2_DIR / "evaluation"

if RUN_EVALUATION:
    experiment2_evaluation_manifest = run_base_vs_lora_evaluation(
        benchmark_path=PROJECT_ROOT / "data" / "benchmark_prompts_v2.jsonl",
        base_model_id="google/flan-t5-base",
        adapter_path=EXPERIMENT2_ADAPTER_PATH,
        output_dir=EXPERIMENT2_EVALUATION_DIR,
        evaluation_config=EvaluationConfig(
            benchmark_path=str(PROJECT_ROOT / "data" / "benchmark_prompts_v2.jsonl"),
            bootstrap_samples=2000,
            seed=52,
        ),
    )
else:
    experiment2_evaluation_manifest = json.loads(
        (EXPERIMENT2_EVALUATION_DIR / "evaluation_manifest.json").read_text(encoding="utf-8")
    )

display(experiment2_evaluation_manifest["comparison"])

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 28209.44it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 12138.86it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | Details
--------------------------+------------+--------
lm_head.bias              | UNEXPECTED |        
lm_head.dense.weight      | UNEXPECTED |        
lm_head.dense.bias        | UNEXPECTED |        
lm_head.layer_norm.weight | UNEXPECTED |        
lm_head.layer_norm.bias   | UNEXPECTED |        
pooler.dense.weight       | MISSING    |        
pooler.dense.bias         | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different tas

{'status': 'completed',
 'benchmark_examples': 80,
 'metric_comparison': {'instruction_adherence': {'base_mean': 0.623099,
   'lora_mean': 0.685447,
   'mean_delta': 0.062349,
   'delta_95_percent_ci': {'mean': 0.062349,
    'lower': 0.042487,
    'upper': 0.082336,
    'n': 80},
   'lora_win_rate': 0.725,
   'tie_rate': 0.0375},
  'quality_rubric_score': {'base_mean': 0.57925,
   'lora_mean': 0.72825,
   'mean_delta': 0.149,
   'delta_95_percent_ci': {'mean': 0.149,
    'lower': 0.116125,
    'upper': 0.183625,
    'n': 80},
   'lora_win_rate': 0.65,
   'tie_rate': 0.2875},
  'rouge_l_f1': {'base_mean': 0.09508,
   'lora_mean': 0.141876,
   'mean_delta': 0.046796,
   'delta_95_percent_ci': {'mean': 0.046796,
    'lower': 0.034206,
    'upper': 0.05976,
    'n': 80},
   'lora_win_rate': 0.7875,
   'tie_rate': 0.0125},
  'semantic_similarity': {'base_mean': 0.346721,
   'lora_mean': 0.48401,
   'mean_delta': 0.137289,
   'delta_95_percent_ci': {'mean': 0.137289,
    'lower': 0.093198,
 

## 10. Compare Experiment 1 directly with Experiment 2

In [19]:
from src.experiment2_comparison import compare_experiment_runs

EXPERIMENT12_COMPARISON_DIR = (
    EXPERIMENT2_EVALUATION_DIR / "experiment1_vs_experiment2"
)

experiment12_summary = compare_experiment_runs(
    experiment1_evaluation_dir=EXPERIMENT1_DIR / "evaluation",
    experiment2_evaluation_dir=EXPERIMENT2_EVALUATION_DIR,
    output_dir=EXPERIMENT12_COMPARISON_DIR,
)
display(experiment12_summary)

experiment12_review_path = (
    EXPERIMENT12_COMPARISON_DIR
    / "experiment1_vs_experiment2_per_example.csv"
)
print("Experiment 1 vs 2 review file:", experiment12_review_path)

{'status': 'completed',
 'benchmark_examples': 80,
 'metric_comparison': {'instruction_adherence': {'experiment1_mean': 0.680236,
   'experiment2_mean': 0.685447,
   'mean_delta': 0.005211,
   'experiment2_win_rate': 0.5,
   'tie_rate': 0.0125},
  'quality_rubric_score': {'experiment1_mean': 0.85425,
   'experiment2_mean': 0.72825,
   'mean_delta': -0.126,
   'experiment2_win_rate': 0.1125,
   'tie_rate': 0.2},
  'rouge_l_f1': {'experiment1_mean': 0.146136,
   'experiment2_mean': 0.141876,
   'mean_delta': -0.00426,
   'experiment2_win_rate': 0.5375,
   'tie_rate': 0.025},
  'semantic_similarity': {'experiment1_mean': 0.530597,
   'experiment2_mean': 0.48401,
   'mean_delta': -0.046587,
   'experiment2_win_rate': 0.3,
   'tie_rate': 0.0125},
  'bertscore_f1': {'experiment1_mean': 0.85392,
   'experiment2_mean': 0.852739,
   'mean_delta': -0.001181,
   'experiment2_win_rate': 0.425,
   'tie_rate': 0.0125}},
 'experiment1_hallucination_flag_rate': 0.5125,
 'experiment2_hallucination_flag

Experiment 1 vs 2 review file: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\evaluation\experiment1_vs_experiment2\experiment1_vs_experiment2_per_example.csv


## 11. Complete Experiment 2 human evaluation

Review:

1. `evaluation/lora_model/manual_review_results.csv`
2. `evaluation/comparison/per_example_base_vs_lora.csv`
3. `evaluation/experiment1_vs_experiment2/experiment1_vs_experiment2_per_example.csv`

For the third file, use only `experiment1`, `experiment2`, or `tie` in
`human_preferred_model`.

In [20]:
exp2_manual_review_path = (
    EXPERIMENT2_EVALUATION_DIR / "lora_model" / "manual_review_results.csv"
)
exp2_base_comparison_path = (
    EXPERIMENT2_EVALUATION_DIR / "comparison" / "per_example_base_vs_lora.csv"
)

print("Experiment 2 model review:", exp2_manual_review_path)
print("Base vs Experiment 2:", exp2_base_comparison_path)
print("Experiment 1 vs Experiment 2:", experiment12_review_path)

comparison_preview = pd.read_csv(experiment12_review_path)
weak_or_flagged = comparison_preview[
    (comparison_preview["delta_exp2_minus_exp1_bertscore_f1"] < 0)
    | (
        comparison_preview["experiment2_hallucination_flag"]
        .astype(str)
        .str.lower()
        .eq("true")
    )
]
display(
    weak_or_flagged[
        [
            "id", "prompt", "experiment1_answer", "experiment2_answer",
            "delta_exp2_minus_exp1_bertscore_f1",
            "experiment2_hallucination_flag",
        ]
    ]
)

Experiment 2 model review: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\evaluation\lora_model\manual_review_results.csv
Base vs Experiment 2: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\evaluation\comparison\per_example_base_vs_lora.csv
Experiment 1 vs Experiment 2: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\evaluation\experiment1_vs_experiment2\experiment1_vs_experiment2_per_example.csv


,id,prompt,experiment1_answer,experiment2_answer,delta_exp2_minus_exp1_bertscore_f1,experiment2_hallucination_flag
0,benchmark_v2_000,Explain supervised learning accurately and inc...,A supervised learning model is a model that le...,A supervised learning model is a model that le...,-0.003051,False
1,benchmark_v2_001,Give a non-confidential quality analytics exam...,Using a manufacturing or process-quality setti...,Modeling supervised learning is a non-confiden...,-0.001734,True
2,benchmark_v2_002,Explain unsupervised learning accurately and i...,Unsupervised learning is a method of learning ...,Unsupervised learning is a method of supervise...,-0.011242,False
3,benchmark_v2_003,Give a non-confidential quality analytics exam...,Unsupervised quality analytics is a method of ...,Unsupervised quality analytics is a non-confid...,0.009449,True
5,benchmark_v2_005,Give a non-confidential quality analytics exam...,Classification is a way of determining the qua...,Modeling is a non-confidential quality analyti...,-0.006894,True
...,...,...,...,...,...,...
73,benchmark_v2_073,Give a non-confidential quality analytics exam...,BERTScore is based on a process-quality settin...,BERTScore is a non-confidential quality analyt...,-0.009253,True
74,benchmark_v2_074,Explain hallucination accurately and include o...,Hallucination is a form of hallucinations. It ...,A hallucination occurs when a person is in a s...,0.016567,True
75,benchmark_v2_075,Give a non-confidential quality analytics exam...,Manufacturing or process-quality is a type of ...,Modeling a manufacturing process is a non-conf...,0.003141,True
77,benchmark_v2_077,Give a non-confidential quality analytics exam...,Model drift occurs when a model drifts in a ma...,Model drift is a non-confidential quality anal...,-0.012344,True


In [21]:
EXPERIMENT2_HUMAN_REVIEW_COMPLETED = True
PROMOTE_EXPERIMENT2 = False

if PROMOTE_EXPERIMENT2 and not EXPERIMENT2_HUMAN_REVIEW_COMPLETED:
    raise RuntimeError(
        "Promotion is blocked until Experiment 2 human review is complete."
    )

print("Experiment 2 human review is complete.")
print("Experiment 2 promotion:", PROMOTE_EXPERIMENT2)

Experiment 2 human review is complete.
Experiment 2 promotion: False


## 12. Apply the Experiment 2 release-quality gate

In [22]:
from src.experiment2_comparison import assess_experiment2_release

release_assessment = assess_experiment2_release(
    experiment2_evaluation_dir=EXPERIMENT2_EVALUATION_DIR,
    experiment_comparison_dir=EXPERIMENT12_COMPARISON_DIR,
)
display(release_assessment)

assessment_path = EXPERIMENT2_DIR / "experiment2_release_assessment.json"
assessment_path.write_text(
    json.dumps(release_assessment, indent=2),
    encoding="utf-8",
)
print("Saved:", assessment_path)

{'ready': False,
 'checks': {'all_exp2_ratings_completed': True,
  'all_exp1_vs_exp2_preferences_completed': True,
  'experiment2_preference_rate_at_least_60_percent': False,
  'mean_factuality_at_least_4': False,
  'mean_relevance_at_least_4': False,
  'mean_clarity_at_least_4': False,
  'mean_instruction_following_at_least_4': False,
  'human_hallucination_rate_below_10_percent': False},
 'experiment2_rating_means': {'human_factuality_1_to_5': 1.4875,
  'human_relevance_1_to_5': 1.875,
  'human_clarity_1_to_5': 2.1375,
  'human_instruction_following_1_to_5': 1.0375},
 'experiment1_vs_experiment2_preferences': {'experiment1': 57,
  'experiment2': 12,
  'tie': 11,
  'missing': 0},
 'experiment2_preference_rate': 0.15,
 'experiment2_human_hallucination_rate': 0.5375,
 'decision': 'do_not_promote_yet'}

Saved: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\GIT Projects\transformer-projects\05-instruction-tuned-domain-llm\outputs\experiments\flan_t5_base_lora_exp2_v3_20260730_135452\experiment2_release_assessment.json


## 13. Promote only after the gate passes

In [23]:
from src.release_utils import promote_experiment

if PROMOTE_EXPERIMENT2:
    if not EXPERIMENT2_HUMAN_REVIEW_COMPLETED:
        raise RuntimeError("Human review is incomplete.")
    if not release_assessment.get("ready", False):
        raise RuntimeError(
            "Experiment 2 did not meet the release-quality thresholds. "
            "Keep it as an experiment and improve the dataset or training."
        )
    release_manifest = promote_experiment(
        project_root=PROJECT_ROOT,
        experiment_dir=EXPERIMENT2_DIR,
        human_review_completed=True,
    )
    display(release_manifest)
else:
    print(
        "Promotion skipped. Experiment 1 remains preserved, and Experiment 2 "
        "remains an evaluated candidate until human review and thresholds pass."
    )

Promotion skipped. Experiment 1 remains preserved, and Experiment 2 remains an evaluated candidate until human review and thresholds pass.


## Final evidence to keep

Commit the code, curated data assets, notebook, dataset quality report, training
curve, metric summaries, comparison charts, and reviewed CSV files. Keep full
adapter ZIP archives outside ordinary Git history or publish the approved
adapter to a Hugging Face model repository.